In [ ]:
from trl import SFTTrainer
from datasets import load_dataset

# load dataset
dataset = load_dataset('trl-lib/Capybara')

# Split into train test
train_dataset = dataset['train']
test_dataset = dataset['test']

In [32]:
# View sample dataset
import random
from pprint import pprint

random_index = random.randint(0, len(train_dataset))
print(f"Previewing sample: {random_index}")
sample = train_dataset[random_index]
pprint(sample)

Previewing sample: 10476
{'messages': [{'content': 'The Federal Reserve System (also known as the '
                          'Federal Reserve or simply the Fed) is the central '
                          'banking system of the United States of America. It '
                          'was created on December 23, 1913, with the '
                          'enactment of the Federal Reserve Act, after a '
                          'series of financial panics (particularly the panic '
                          'of 1907) led to the desire for central control of '
                          'the monetary system in order to alleviate financial '
                          'crises. Over the years, events such as the Great '
                          'Depression in the 1930s and the Great Recession '
                          'during the 2000s have led to the expansion of the '
                          'roles and responsibilities of the Federal Reserve '
                          'System.\\n\\nA

In [27]:
print(f"Number of train samples: {len(train_dataset)}")
print(f"Number of test samples: {len(test_dataset)}")

Number of train samples: 15806
Number of test samples: 200


In [ ]:
# Load Model
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = 'Qwen/Qwen3-0.6B'
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 1024)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
        (post_attention_layer

### Evaluate the model before finetuning

In [28]:
from trl import SFTTrainer, SFTConfig

eval_args = SFTConfig(
    output_dir='./eval-output',
    per_device_eval_batch_size=2,
    report_to='none'
)

trainer = SFTTrainer(
    model=model_name,
    args=eval_args,
    train_dataset=test_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer
)

metrics = trainer.evaluate()
print(metrics)

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Tokenizing train dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Training Loss,Validation Loss,Step,Entropy,Num Tokens,Mean Token Accuracy
No log,2.039165,0,1.272083,0.000000,0.599440


{'eval_loss': 2.0391645431518555, 'eval_entropy': 1.272083079814911, 'eval_num_tokens': 0.0, 'eval_mean_token_accuracy': 0.5994395962357522}


#### Finetune without LoRA

In [46]:
args = SFTConfig(
    output_dir = "./sft_output",
    eval_strategy = 'steps',
    eval_steps = 100,
    logging_steps = 20,
    per_device_eval_batch_size = 2,
    per_device_train_batch_size = 4
)

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer
)

# train the model
trainer.train()
# evaluate
metrics = trainer.evaluate()
print(metrics)

Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
100,1.524473,1.521967,1.476419,282567.000000,0.640013
200,1.466526,1.504381,1.456083,559789.000000,0.642968
300,1.561328,1.496233,1.431578,846716.000000,0.645136
400,1.598616,1.488694,1.442786,1140680.000000,0.646175
500,1.538654,1.481995,1.455753,1427210.000000,0.647540
600,1.443554,1.477604,1.476007,1722987.000000,0.648052
700,1.445332,1.474076,1.450369,1996601.000000,0.648312
800,1.542477,1.468568,1.433497,2284597.000000,0.649916
900,1.482281,1.466680,1.429943,2571400.000000,0.650210
1000,1.412499,1.464804,1.419385,2853606.000000,0.649970


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (16 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

RuntimeError: [enforce fail at inline_container.cc:672] . unexpected pos 2189055744 vs 2189055632

In [57]:
import pandas as pd

queries = []
expected_responses = []

for idx, sample in enumerate(test_dataset):
    queries.extend([query['content'] for query in sample['messages'] if query['role'] == 'user'])
    expected_responses.extend([query['content'] for query in sample['messages'] if query['role'] == 'assistant'])

print(f"Total number of queries: {len(queries)}")
print(f"Total number of expected answers: {len(expected_responses)}")

#### Print some samples
print(queries[0])
print(expected_responses[0])

Total number of queries: 509
Total number of expected answers: 509
How can I innovate today?
To innovate today, you can follow these steps:

1. Identify a problem or opportunity: Start by considering the challenges you face in your daily life, workplace, or industry. Look for opportunities to improve existing processes or create new solutions.

2. Research and learn: Gather information about the problem or opportunity. Read articles, watch videos, listen to podcasts, and talk to people who have experience in the area. This will give you a better understanding of the issue and potential solutions.

3. Brainstorm ideas: Generate as many ideas as possible, without worrying about their feasibility or quality at this stage. Encourage "out-of-the-box" thinking and consider various perspectives.

4. Analyze and refine your ideas: Evaluate your ideas based on their feasibility, potential impact, and alignment with your goals. Keep refining and combining ideas to develop innovative solutions.



In [ ]:
import torch

def test_model(model, test_dataset):
    
    max_length = 1024

    for query, expected_response in zip(queries, expected_responses):
        # messages = sample #sample['messages']

        # Apply chat template and tokenize
        text = tokenizer.apply_chat_template(
            query,
            add_generation_prompt = True, # Add assistant prompt
            tokenize=False # Return strings, not tokens
        )

        # Tokenize and move to appropriate device
        inputs = tokenizer(text, return_tensors='pt').to(model.device)

        print(f"Processing: {query}")

        with torch.inference_mode():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_length,
                temperature=0.1,
                do_sample=True,
                top_p=0.9,
                pad_token_id=tokenizer.eos_token_id,
                repetition_penalty=1.1,
                length_penalty=1.0
            )
        # Decode and extract only the generated portion
        response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        print(response)
        break

Processing: How can I innovate today?


### **Finetune with LoRA**

In [ ]:
# Load Model
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model


model_name = 'Qwen/Qwen3-0.6B'
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

peft_config = LoraConfig(
    r=32,
    lora_alpha=6,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj']
)

model = get_peft_model(model, peft_config)

training_args = SFTConfig(
    output_dir = "./sft_lora_output",
    eval_strategy = 'steps',
    eval_steps = 20,
    logging_steps = 20,
    max_steps=200,
    per_device_eval_batch_size = 4,
    per_device_train_batch_size = 8,
    learning_rate=2.0e-4, # use 2.0e-5 for full fine-tuning
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer
)

trainer.train()

metrics = trainer.evaluate()
print(metrics)


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
20,1.870901,1.650012,1.561670,114581.000000,0.620334
40,1.581353,1.592016,1.547528,224687.000000,0.630304
60,1.617911,1.567490,1.524626,336705.000000,0.634171
80,1.517015,1.556815,1.506282,448766.000000,0.635354
100,1.499864,1.550676,1.493876,559789.000000,0.636347
120,1.439819,1.546499,1.496225,674400.000000,0.636990
140,1.577666,1.543508,1.510177,790088.000000,0.637412
160,1.538768,1.541411,1.501356,909154.000000,0.637938
180,1.558445,1.540083,1.505278,1022407.000000,0.638110
200,1.619850,1.539915,1.510851,1140680.000000,0.637991


Training Loss,Validation Loss,Step,Entropy,Num Tokens,Mean Token Accuracy
1.619850,1.539915,200,1.510851,1140680.000000,0.637991


{'eval_loss': 1.539914846420288, 'eval_entropy': 1.510850967168808, 'eval_num_tokens': 1140680.0, 'eval_mean_token_accuracy': 0.6379914772510529}


### **Using QLoRA**

In [ ]:
import torch
from peft import LoraConfig
from transformers import (
    BitsAndBytesConfig,
    AutoModelForCausalLM,
    AutoTokenizer
)
from trl import SFTTrainer, SFTConfig


model_name = 'Qwen/Qwen3-0.6B'
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Configure 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4', # or "fp4"
    bnb_4bit_compute_dtype=torch.bfloat16, # Compute dtype for 4-bit base models
    bnb_4bit_use_double_quant=True # Nested quantization for additional memory savings
)

# Configure LoRA
peft_config = LoraConfig(
    r=32,
    lora_alpha=16,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM'
)

# Configure training with higher learning rate for LoRA
training_args = SFTConfig(
    learning_rate=2.0e-4, # 10x the base rate for QLoRA
)

# Create trainer with quantization and PEFT config
trainer = SFTTrainer(
    model='meta-llama/Llama-2-7b-hf',
    args=training_args,
    train_dataset=train_dataset,
    quantization_config=bnb_config,
    peft_config=peft_config
)

trainer.train()

### **SmoLM Finetuning**

In [ ]:
base_tokenizer.apply_chat_template()

TokenizersBackend(name_or_path='HuggingFaceTB/SmolLM3-3B-Base', vocab_size=128000, model_max_length=131072, padding_side='left', truncation_side='right', special_tokens={'eos_token': '<|end_of_text|>'}, added_tokens_decoder={
	128000: AddedToken("<|begin_of_text|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	128001: AddedToken("<|end_of_text|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	128002: AddedToken("<think>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=False),
	128003: AddedToken("</think>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=False),
	128004: AddedToken("<|finetune_right_pad_id|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	128005: AddedToken("<|reserved_special_token_2|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	128006: AddedToken("<|start_header_id|>", rstri

In [ ]:
# Compare the based model to the fine-tuned Model
from transformers import pipeline, AutoTokenizer

model_id = "HuggingFaceTB/SmolLM3-3B-Base"

base_tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM3-3B")

# Initialize the pipeline
pipe = pipeline(
    'text-generation', 
    model=model_id, # Base Model
    tokenizer=base_tokenizer,
    device_map='auto'
    )

# Define conversation
messages = [
    {'role': 'system', 'content': "you are a friendly chatbot who always responds in the style of a pirate"},
    {'role': 'user', 'content': 'how many helicopters can a human eat in one sitting?'}
]

# Generate response - pipeline handles chat templates automatically
response = pipe(
    messages, 
    max_new_tokens=128,
    do_sample=True,
    temperature=0.7
    )

print(response[0]['generated_text'][-1])

# Output
# {'role': 'assistant', 'content': "Well, the amount of helicopters a human can eat in one sitting depends on the size of the helicopter and the person's appetite. 
# Typically, a regular-sized human can only consume a single helicopter in one sitting without any complications. 
# However, larger helicopters or those made out of a more digestible material might require a bit more time for a human to eat.
# \n\nAs for the number, there are a few things to consider. Firstly, the size of the helicopter could affect how much a person can consume in one sitting. 
# A small helicopter might take up less space in the stomach and could be more easily digested, while a larger helicopter could cause"}

Loading weights:   0%|          | 0/326 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


{'role': 'assistant', 'content': "Well, the amount of helicopters a human can eat in one sitting depends on the size of the helicopter and the person's appetite. Typically, a regular-sized human can only consume a single helicopter in one sitting without any complications. However, larger helicopters or those made out of a more digestible material might require a bit more time for a human to eat.\n\nAs for the number, there are a few things to consider. Firstly, the size of the helicopter could affect how much a person can consume in one sitting. A small helicopter might take up less space in the stomach and could be more easily digested, while a larger helicopter could cause"}


In [ ]:
# Try a Instruction Tuned Model
from transformers import pipeline


# Initialize the pipeline
pipe = pipeline(
    'text-generation', 
    "HuggingFaceTB/SmolLM3-3B",
    device_map='auto'
    )

# Define conversation
messages = [
    {'role': 'system', 'content': "you are a friendly chatbot who always responds in the style of a pirate"},
    {'role': 'user', 'content': 'how many helicopters can a human eat in one sitting?'}
]

# Generate response - pipeline handles chat templates automatically
response = pipe(messages, max_new_tokens = 128, temperature=0.7)
print(response[0]['generated_text'][-1])

# Output
# {'role': 'assistant', 'content': '<think>\nOkay, the user is asking, "how many helicopters can a human eat in one sitting?" That\'s a pretty strange question. Let me break it down. First, helicopters are large, metal aircraft with rotors. They\'re not edible. Humans can\'t eat helicopters because they\'re not food. But maybe the user is trying to be funny or is confused.\n\nI need to respond in a pirate-style. Pirates are known for their colorful language and humor. So I should use a pirate\'s tone with some humor and maybe a bit of sarcasm. The key is to address the question while keeping it light-hearted and in character'}

model.safetensors.index.json:   0%|          | 0.00/26.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

/home/monster/Desktop/codes/ML-Workbook/myvenv/lib/python3.12/site-packages/huggingface_hub/file_download.py:773: UserWarning: Not enough free disk space to download the file. The expected file size is: 4966.32 MB. The target location /home/monster/.cache/huggingface/hub/models--HuggingFaceTB--SmolLM3-3B/blobs only has 4584.49 MB free disk space.
  warnings.warn(
/home/monster/Desktop/codes/ML-Workbook/myvenv/lib/python3.12/site-packages/huggingface_hub/file_download.py:773: UserWarning: Not enough free disk space to download the file. The expected file size is: 4966.32 MB. The target location /home/monster/.cache/huggingface/hub/models--HuggingFaceTB--SmolLM3-3B/blobs only has 4584.48 MB free disk space.
  warnings.warn(


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

... (7 outputs left out)

special_tokens_map.json:   0%|          | 0.00/289 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/5.60k [00:00<?, ?B/s]

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


{'role': 'assistant', 'content': '<think>\nOkay, the user is asking, "how many helicopters can a human eat in one sitting?" That\'s a pretty strange question. Let me break it down. First, helicopters are large, metal aircraft with rotors. They\'re not edible. Humans can\'t eat helicopters because they\'re not food. But maybe the user is trying to be funny or is confused.\n\nI need to respond in a pirate-style. Pirates are known for their colorful language and humor. So I should use a pirate\'s tone with some humor and maybe a bit of sarcasm. The key is to address the question while keeping it light-hearted and in character'}


In [5]:
# Initialize the pipeline
pipe = pipeline(
    'text-generation', 
    "HuggingFaceTB/SmolLM3-3B",
    device_map='auto'
    )


# Configure generation parameters
generation_config = {
    "max_new_tokens": 200,
    "temperature": 0.8,
    "do_sample": True,
    "top_p": 0.9,
    "repetition_penalty": 1.1
}

# Multi-turn conversation
conversation = [
    {"role": "system", "content": "You are a helpful math tutor."},
    {"role": "user", "content": "Can you help me with calculus?"},
]

# Generate first response
response = pipe(conversation, **generation_config)
conversation = response[0]['generated_text']

# Continue the conversation
conversation.append({"role": "user", "content": "What is a derivative?"})
response = pipe(conversation, **generation_config)

print("Final conversation:")
for message in response[0]['generated_text']:
    print(f"{message['role']}: {message['content']}")

Loading weights:   0%|          | 0/326 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.


Final conversation:
system: You are a helpful math tutor.
user: Can you help me with calculus?
assistant: <think>
Okay, the user is asking for help with calculus. Let me start by understanding what they need. They might be struggling with a specific topic like limits, derivatives, integrals, or maybe something more advanced like differential equations or multivariable calculus. Since they didn't specify, I should ask them to clarify which area they're having trouble with.

But wait, maybe they just want some general guidance on how to approach calculus problems. I can offer tips on studying techniques, resources, or common mistakes to avoid. However, without knowing their exact needs, it's hard to give targeted help. So, I'll probably respond by asking them to specify which part of calculus they're working on and any particular questions or topics that are giving them trouble.

Also, considering different learning styles, I could suggest breaking down complex problems into smaller step

In [8]:
# Using Tokenizer with chat template

from transformers import AutoTokenizer

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM3-3B")

# Structure your conversation as a list of message dictionaries
messages = [
    {"role": "system", "content": "You are a helpful assistant focused on technical topics."},
    {"role": "user", "content": "Can you explain what a chat template is?"},
    {"role": "assistant", "content": "A chat template structures conversations between users and AI models by providing a consistent format that helps the model understand different roles and maintain context."}
]

# Apply chat template
formatted_chat = tokenizer.apply_chat_template(
    messages,
    tokenize=False, # Return string instead of tokens
    add_generation_prompt=True # Add prompt for next assistant response
)
print(formatted_chat)

<|im_start|>system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 30 October 2026
Reasoning Mode: /think

## Custom Instructions

You are a helpful assistant focused on technical topics.

<|im_start|>user
Can you explain what a chat template is?<|im_end|>
<|im_start|>assistant
A chat template structures conversations between users and AI models by providing a consistent format that helps the model understand different roles and maintain context.<|im_end|>
<|im_start|>assistant



In [ ]:
# Professional assistant
system_message = {
    "role": "system",
    "content": "You are a professional customer service agent. Always be polite, clear, and helpful."
}

# Technical expert
system_message = {
    "role": "system",
    "content": "You are a senior software engineer. Provide detailed technical explanations with code examples when appropriate."
}

# Creative assistant
system_message = {
    "role": "system",
    "content": "You are a creative writing assistant. Help users craft engaging stories and provide constructive feedback."
}

# Multi-Turn Conversation
conversation = [
    {"role": "system", "content": "You are a helpful programming tutor."},
    {"role": "user", "content": "I'm learning Python. Can you explain functions?"},
    {"role": "assistant", "content": "Functions in Python are reusable blocks of code that perform specific tasks. They're defined using the 'def' keyword."},
    {"role": "user", "content": "Can you show me an example?"},
    {"role": "assistant", "content": "Sure! Here's a simple function:\n\n```python\ndef greet(name):\n    return f'Hello, {name}!'\n\nresult = greet('Alice')\nprint(result)  # Output: Hello, Alice!\n```"},
    {"role": "user", "content": "How do I make it return multiple values?"},
]

In [10]:
def create_thinking_example(question, answer, reasoning=None):
    """Create a training example with optional thinking"""
    if reasoning:
        assistant_content = f"<|thinking|>\n{reasoning}\n</|thinking|>\n\n{answer}"
    else:
        assistant_content = answer
    
    return [
        {"role": "user", "content": question},
        {"role": "assistant", "content": assistant_content}
    ]

# Example usage
math_example = create_thinking_example(
    question="What is the derivative of x²?",
    answer="The derivative of x² is 2x",
    reasoning="Using the power rule: d/dx(x^n) = n·x^(n-1)\nFor x²: n=2, so d/dx(x²) = 2·x^(2-1) = 2x"
)

print(f"math reasoning example: {math_example}")

math reasoning example: [{'role': 'user', 'content': 'What is the derivative of x²?'}, {'role': 'assistant', 'content': '<|thinking|>\nUsing the power rule: d/dx(x^n) = n·x^(n-1)\nFor x²: n=2, so d/dx(x²) = 2·x^(2-1) = 2x\n</|thinking|>\n\nThe derivative of x² is 2x'}]


In [11]:
# Conversation with tool usage

# Define available tools
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather for a location",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "The city and state, e.g. San Francisco, CA"
                    },
                    "unit": {
                        "type": "string",
                        "enum": ["celsius", "fahrenheit"],
                        "description": "The temperature unit"
                    }
                },
                "required": ["location"]
            }
        }
    },
    {
        "type": "function", 
        "function": {
            "name": "calculate",
            "description": "Perform mathematical calculations",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "Mathematical expression to evaluate"
                    }
                },
                "required": ["expression"]
            }
        }
    }
]
messages = [
    {"role": "system", "content": "You are a helpful assistant with access to tools."},
    {"role": "user", "content": "What's the weather like in Paris?"},
    {
        "role": "assistant", 
        "content": "I'll check the weather in Paris for you.",
        "tool_calls": [
            {
                "id": "call_1",
                "type": "function",
                "function": {
                    "name": "get_weather",
                    "arguments": '{"location": "Paris, France", "unit": "celsius"}'
                }
            }
        ]
    },
    {
        "role": "tool",
        "tool_call_id": "call_1", 
        "content": '{"temperature": 22, "condition": "sunny", "humidity": 60}'
    },
    {
        "role": "assistant",
        "content": "The weather in Paris is currently sunny with a temperature of 22°C and 60% humidity. It's a beautiful day!"
    }
]

# Apply chat template with tools
formatted_with_tools = tokenizer.apply_chat_template(
    messages,
    tools=tools,
    tokenize=False,
    add_generation_prompt=False
)

print("Chat template with tools:")
print(formatted_with_tools)

Chat template with tools:
<|im_start|>system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 30 October 2026
Reasoning Mode: /think

## Custom Instructions

You are a helpful assistant with access to tools.

### Tools

You may call one or more functions to assist with the user query.
You are provided with function signatures within <tools></tools> XML tags:

<tools>
{'type': 'function', 'function': {'name': 'get_weather', 'description': 'Get the current weather for a location', 'parameters': {'type': 'object', 'properties': {'location': {'type': 'string', 'description': 'The city and state, e.g. San Francisco, CA'}, 'unit': {'type': 'string', 'enum': ['celsius', 'fahrenheit'], 'description': 'The temperature unit'}}, 'required': ['location']}}}
{'type': 'function', 'function': {'name': 'calculate', 'description': 'Perform mathematical calculations', 'parameters': {'type': 'object', 'properties': {'expression': {'type': 'string', 'description': 'Mathematical expression to eval

In [ ]:
# Training with Tool Usage
def format_tool_dataset(examples):
    """Format dataset with tool usage for training"""
    formatted_texts = []

    for messages, tools in zip(examples['messages'], examples.get('tools'), [None] * len(examples['messages'])):
        formatted_text = tokenizer.apply_chat_template(
            messages,
            tools=tools,
            tokenize=False,
            add_generation_prompt=False
        )
        formatted_texts.append(formatted_text)
    return {'text': formatted_texts}

In [12]:
# Inspecting a Model's Chat Template

# View the actual template
print("SmolLM3 Chat Template")
print(tokenizer.chat_template)

# See what special tokens are used
print("\nSpecial tokens:")
print(f"BOS: {tokenizer.bos_token}")
print(f"EOS: {tokenizer.eos_token}")
print(f"UNK: {tokenizer.unk_token}")
print(f"PAD: {tokenizer.pad_token}")

# check for custom tokens
special_tokens = tokenizer.special_tokens_map
for name, token in special_tokens.items():
    print(f"{name}: {token}")

SmolLM3 Chat Template
{# ───── defaults ───── #}
{%- if enable_thinking is not defined -%}
{%- set enable_thinking = true -%}
{%- endif -%}

{# ───── reasoning mode ───── #}
{%- if enable_thinking -%}
  {%- set reasoning_mode = "/think" -%}
{%- else -%}
  {%- set reasoning_mode = "/no_think" -%}
{%- endif -%}

{# ───── header (system message) ───── #}
{{- "<|im_start|>system\n" -}}

{%- if messages[0].role == "system" -%}
  {%- set system_message = messages[0].content -%}
  {%- if "/no_think" in system_message -%}
    {%- set reasoning_mode = "/no_think" -%}
  {%- elif "/think" in system_message -%}
    {%- set reasoning_mode = "/think" -%}
  {%- endif -%}
  {%- set custom_instructions = system_message.replace("/no_think", "").replace("/think", "").rstrip() -%}
{%- endif -%}

{%- if "/system_override" in system_message -%}
  {{- custom_instructions.replace("/system_override", "").rstrip() -}}
  {{- "<|im_end|>\n" -}}
{%- else -%}
  {{- "## Metadata\n\n" -}}
  {{- "Knowledge Cutoff Date

In [ ]:
# Create a custom template (advanced users only)
custom_template = """
{%- for message in messages %}
    {%- if message['role'] == 'system' %}
        {%- set system_message = message['content'] %}
    {%- endif %}
{%- endfor %}
{%- if system_message is defined %}
<|system|>{{ system_message }}<|end|>
{%- endif %}
{%- for message in messages %}
    {%- if message['role'] == 'user' %}
<|user|>{{ message['content'] }}<|end|>
    {%- elif message['role'] == 'assistant' %}
<|assistant|>{{ message['content'] }}<|end|>
    {%- endif %}
{%- endfor %}
{%- if add_generation_prompt %}
<|assistant|>
{%- endif %}
"""

# Apply custom template (be very careful with this!)
# tokenizer.chat_template = custom_template

In [13]:
# Template debugging
def debug_chat_template(messages, tokenizer):
    """Debug chat template application"""
    
    # Apply template
    formatted = tokenizer.apply_chat_template(
        messages, 
        tokenize=False, 
        add_generation_prompt=True
    )
    
    # Tokenize and decode to see actual tokens
    tokens = tokenizer(formatted, return_tensors="pt")
    
    print("=== TEMPLATE DEBUG ===")
    print(f"Input messages: {len(messages)}")
    print(f"Formatted length: {len(formatted)} chars")
    print(f"Token count: {tokens['input_ids'].shape[1]}")
    print("\nFormatted text:")
    print(repr(formatted))  # Shows escape characters
    print("\nTokens:")
    print(tokens['input_ids'][0].tolist()[:20], "...")  # First 20 tokens
    print("\nDecoded tokens:")
    for i, token_id in enumerate(tokens['input_ids'][0][:20]):
        token = tokenizer.decode([token_id])
        print(f"{i:2d}: {token_id:5d} -> {repr(token)}")

# Example usage
debug_messages = [
    {"role": "user", "content": "Hello!"},
    {"role": "assistant", "content": "Hi there!"}
]

debug_chat_template(debug_messages, tokenizer)

=== TEMPLATE DEBUG ===
Input messages: 2
Formatted length: 1403 chars
Token count: 258

Formatted text:
'<|im_start|>system\n## Metadata\n\nKnowledge Cutoff Date: June 2025\nToday Date: 30 October 2026\nReasoning Mode: /think\n\n## Custom Instructions\n\nYou are a helpful AI assistant named SmolLM, trained by Hugging Face. Your role as an assistant involves thoroughly exploring questions through a systematic thinking process before providing the final precise and accurate solutions. This requires engaging in a comprehensive cycle of analysis, summarizing, exploration, reassessment, reflection, backtracking, and iteration to develop well-considered thinking process. Please structure your response into two main sections: Thought and Solution using the specified format: <think> Thought section </think> Solution section. In the Thought section, detail your reasoning process in steps. Each step should include detailed considerations such as analysing questions, summarizing relevant findings

In [ ]:
# Training COnfiguration

learning_rate = 5e-5 # to [5e-5 (stable, slower), 1e-4 (recommended), 2e-4 (faster, less stable)]
per_device_train_batch_size = 4 # [2 (limited GPU Memory), 4 (Balanced GPU Memory), 8 (More GPU Memory)]
gradient_accumulation_steps = 8 # [8 (limited GPU Memory), 4 (Balanced GPU Memory), 2 (More GPU Memory)]
max_seq_len = 2048 # [512 (very short sequences), 1024 (Short sequences), 2048 (Long sequences), 4096 (Very long sequences)]
training_steps = 1000 # to 5000
warmUp_steps = 0.1 # 10% of total


In [ ]:
# Simple Trackio integration
import trackio
import os

os.environ["TRACKIO_PROJECT_NAME"] = "my-project"
os.environ["TRACKIO_SPACE_ID"] = "username/space_id"

# Initialize tracking
trackio.init(project="smollm3-sft")

# Log metrics during training
trackio.log({"train_in_loss": 0.5, "learning_rate": 5e-5})

# Finish tracking
trackio.finish()

from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    args=config,
)

In [ ]:
### Expected Dataset type and format

# Standard language modeling
{"text": "The sky is blue."}

# Conversational language modeling
{"messages": [{"role": "user", "content": "What color is the sky?"},
              {"role": "assistant", "content": "It is blue."}]}

# Standard prompt-completion
{"prompt": "The sky is",
 "completion": " blue."}

# Conversational prompt-completion
{"prompt": [{"role": "user", "content": "What color is the sky?"}],
 "completion": [{"role": "assistant", "content": "It is blue."}]}

In [6]:
### Formatting dataset

from datasets import load_dataset

dataset = load_dataset("FreedomIntelligence/medical-o1-reasoning-SFT", "en")

def preprocess_function(example):
    return {
        "prompt": [{"role": "user", "content": example["Question"]}],
        "completion": [
            {"role": "assistant", "content": f"<think>{example['Complex_CoT']}</think>{example['Response']}"}
        ],
    }

dataset = dataset.map(preprocess_function, remove_columns=["Question", "Response", "Complex_CoT"])
print(next(iter(dataset["train"])))

README.md:   0%|          | 0.00/1.97k [00:00<?, ?B/s]

medical_o1_sft.json: reconstructing file:   0%|          |  0.00B / 58.2MB            

medical_o1_sft.json: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/19704 [00:00<?, ? examples/s]

Map:   0%|          | 0/19704 [00:00<?, ? examples/s]

{'prompt': [{'role': 'user', 'content': 'Given the symptoms of sudden weakness in the left arm and leg, recent long-distance travel, and the presence of swollen and tender right lower leg, what specific cardiac abnormality is most likely to be found upon further evaluation that could explain these findings?'}], 'completion': [{'role': 'assistant', 'content': "<think>Okay, let's see what's going on here. We've got sudden weakness in the person's left arm and leg - and that screams something neuro-related, maybe a stroke?\n\nBut wait, there's more. The right lower leg is swollen and tender, which is like waving a big flag for deep vein thrombosis, especially after a long flight or sitting around a lot.\n\nSo, now I'm thinking, how could a clot in the leg end up causing issues like weakness or stroke symptoms?\n\nOh, right! There's this thing called a paradoxical embolism. It can happen if there's some kind of short circuit in the heart - like a hole that shouldn't be there.\n\nLet's put 

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset
import trackio as wandb
import torch
from peft import PeftModel


model_id = 'HuggingFaceTB/SmolLM3-3B-Base'

# Load SmolLM3 base model
base_model = AutoModelForCausalLM.from_pretrained('HuggingFaceTB/SmolLM3-3B-Base')
tokenizer = AutoTokenizer.from_pretrained('HuggingFaceTB/SmolLM3-3B-Base')


# Load dataset
dataset = load_dataset('HuggingFaceTB/smoltalk2_everyday_convs_think')

# Configure training with Trackio integration
config = SFTConfig(
    output_dir='./smollm3_finetuned',
    per_device_train_batch_size=1,
    learning_rate=5e-5,
    max_steps=1000,
    report_to='trackio'
)

trainer = SFTTrainer(
    model=model_id,
    train_dataset=dataset['train'],
    args=config,
    # quantization_config=bnb_config
)

# Initialize experiment tracking
wandb.init(project='smollm3-sft', name='sft-run')

trainer.train()

Loading weights:   0%|          | 0/326 [00:00<?, ?it/s]

* Trackio project initialized: smollm3-sft
* Trackio metrics logged to: /home/monster/.cache/huggingface/trackio
* NVIDIA GPU detected, enabling automatic GPU metrics logging
* psutil detected, enabling automatic CPU/system metrics logging
* Created new run: sft-run


* Run finished. Uploading logs to Trackio (please wait...)
* Trackio project initialized: huggingface
* Trackio metrics logged to: /home/monster/.cache/huggingface/trackio
* NVIDIA GPU detected, enabling automatic GPU metrics logging
* psutil detected, enabling automatic CPU/system metrics logging
* Created new run: calm-river-2


OutOfMemoryError: CUDA out of memory. Tried to allocate 22.00 MiB. GPU 0 has a total capacity of 15.61 GiB of which 6.69 MiB is free. Including non-PyTorch memory, this process has 15.36 GiB memory in use. Of the allocated memory 14.99 GiB is allocated by PyTorch, and 136.27 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [1]:

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset
import trackio as wandb
import torch
from peft import PeftModel, LoraConfig

# Configure 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4', # or "fp4"
    bnb_4bit_compute_dtype=torch.bfloat16, # Compute dtype for 4-bit base models
    bnb_4bit_use_double_quant=True # Nested quantization for additional memory savings
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj']#'all_linear'
)

model_id = 'HuggingFaceTB/SmolLM3-3B-Base'

# Load SmolLM3 base model
# base_model = AutoModelForCausalLM.from_pretrained('HuggingFaceTB/SmolLM3-3B-Base')
tokenizer = AutoTokenizer.from_pretrained('HuggingFaceTB/SmolLM3-3B-Base')

# Load dataset
dataset = load_dataset('HuggingFaceTB/smoltalk2_everyday_convs_think')

# Configure training with Trackio integration
config = SFTConfig(
    output_dir='./smollm3_finetuned',
    num_train_epochs=1,
    packing=True,
    per_device_train_batch_size=4,
    learning_rate=5e-5,
    max_steps=1000,
    report_to='trackio'
)

trainer = SFTTrainer(
    model=model_id,
    train_dataset=dataset['train'],
    args=config,
    peft_config=lora_config,
    quantization_config=bnb_config
)

# Initialize experiment tracking
# wandb.init(project='smollm3-sft', name='sft-run')

trainer.train()

Loading weights:   0%|          | 0/326 [00:00<?, ?it/s]

* Trackio project initialized: huggingface
* Trackio metrics logged to: /home/monster/.cache/huggingface/trackio
* NVIDIA GPU detected, enabling automatic GPU metrics logging
* psutil detected, enabling automatic CPU/system metrics logging
* Created new run: calm-river-2


Step,Training Loss
10,1.847738
20,1.695600
30,1.557509
40,1.363747
50,1.087334
60,0.903116
70,0.895287
80,0.854092
90,0.841947
100,0.829346


* Run finished. Uploading logs to Trackio (please wait...)


TrainOutput(global_step=1000, training_loss=0.7809126586914062, metrics={'train_runtime': 3634.5637, 'train_samples_per_second': 1.101, 'train_steps_per_second': 0.275, 'total_flos': 6.927207626283418e+16, 'train_loss': 0.7809126586914062, 'epoch': 2.159827213822894})

In [3]:
# Merging Implementation
import torch
from transformers import AutoModelForCausalLM
from peft import PeftModel

# 1. Load base model
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.bfloat16,
    device_map='auto'
)

# 2. Load the PEFT model with adapter
peft_model = PeftModel.from_pretrained(
    base_model,
    './smollm3_finetuned/checkpoint-1000',
    dtype=torch.bfloat16
)

# 3. Merge adapter weights with base model
try:
    merged_model = peft_model.merge_and_unload()
except RuntimeError as e:
    print(f"Merging failed: {e}")
    # implement fall back

# 4. Save the merged model
merged_model_path = './smollm3_finetuned_merged'

merged_model.save_pretrained(merged_model_path)
tokenizer.save_pretrained(merged_model_path)


Loading weights:   0%|          | 0/326 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./smollm3_finetuned_merged/tokenizer_config.json',
 './smollm3_finetuned_merged/tokenizer.json')

In [5]:
# Compare the based model to the fine-tuned Model
from transformers import pipeline, AutoTokenizer

model_id = './smollm3_finetuned_merged' # "HuggingFaceTB/SmolLM3-3B-Base"

base_tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM3-3B")

# Initialize the pipeline
pipe = pipeline(
    'text-generation', 
    model=model_id, # Base Model
    tokenizer=base_tokenizer,
    device_map='auto'
    )

# Define conversation
messages = [
    {'role': 'system', 'content': "you are a friendly chatbot who always responds in the style of a pirate"},
    {'role': 'user', 'content': 'how many helicopters can a human eat in one sitting?'}
]

# Generate response - pipeline handles chat templates automatically
response = pipe(
    messages, 
    max_new_tokens=128,
    do_sample=True,
    temperature=0.7
    )

print(response[0]['generated_text'][-1])

Loading weights:   0%|          | 0/326 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


{'role': 'assistant', 'content': '<think>\nOkay, let\'s try to figure out how many helicopters a human can eat in one sitting. Hmm, first, I need to clarify the question. Is it about the physical number of helicopters, or is it a joke? The user might be testing my understanding of the phrase "eat one sitting." Since I can\'t eat real helicopters, maybe it\'s a play on words or a riddle. But the user is asking seriously, so I should consider different possibilities.\n\nWait, maybe they\'re referring to a specific type of helicopter, like a human-shaped helicopter? That doesn\'t make much sense. Or perhaps it\'s a parody'}
